# Embed `joint_dataset` nodes with voyage-4-large

Embeds `title + "\n\n" + abstract` for all 384,756 nodes of `joint_dataset/nodes.csv`.

**Output:** `joint_dataset/node_embeddings.npy` — float32 array of shape `(384756, D)`.
Row `i` is the embedding of `node_id == i` (verified below), so it loads with
`np.load(...)` and indexes directly by `node_id`.

**Crash safety:** embeddings are written straight into a memory-mapped `.npy` on disk and
`node_embeddings.progress.json` records how many rows are finished. Re-running this notebook
resumes from there instead of starting over. A batch that keeps failing after all retries raises,
so the run stops with everything completed so far already on disk.

In [1]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import voyageai
from dotenv import load_dotenv

MODEL      = "voyage-4-large"
INPUT_TYPE = "document"
BATCH_SIZE = 128          # Voyage max texts per request
MAX_RETRIES = 6

ROOT      = Path.cwd().parent
OUT_PATH  = ROOT / "joint_dataset" / "node_embeddings.npy"
PROG_PATH = ROOT / "joint_dataset" / "node_embeddings.progress.json"

load_dotenv(ROOT / ".env")
client = voyageai.Client(api_key=os.environ["VOYAGE_API_KEY"])
print("Voyage client ready")

/opt/anaconda3/envs/DyTAG/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Voyage client ready


In [2]:
nodes = pd.read_csv(ROOT / "joint_dataset" / "nodes.csv", usecols=["node_id", "title", "abstract"])

# Row index must equal node_id, otherwise the saved array can't be indexed by node_id.
assert (nodes["node_id"].to_numpy() == np.arange(len(nodes))).all(), "node_id is not 0..N-1 in row order"

texts = (nodes["title"].fillna("") + "\n\n" + nodes["abstract"].fillna("")).tolist()
texts = [t if t.strip() else " " for t in texts]   # Voyage rejects empty strings
N = len(texts)
print(f"{N:,} nodes loaded")

384,756 nodes loaded


In [3]:
def embed_batch(batch):
    """Embed one batch, retrying with exponential backoff. Raises if it never succeeds."""
    for attempt in range(MAX_RETRIES):
        try:
            return client.embed(batch, model=MODEL, input_type=INPUT_TYPE).embeddings
        except Exception as e:
            wait = 2 ** attempt
            print(f"    {type(e).__name__}: {e} — retrying in {wait}s ({attempt + 1}/{MAX_RETRIES})", flush=True)
            time.sleep(wait)
    raise RuntimeError(f"batch failed after {MAX_RETRIES} attempts")

In [4]:
# Resume from a previous run if there is one.
if PROG_PATH.exists():
    prog = json.loads(PROG_PATH.read_text())
    assert prog["model"] == MODEL, f"existing run used {prog['model']}, not {MODEL}"
    rows_done = prog["rows_done"]
    emb = np.lib.format.open_memmap(OUT_PATH, mode="r+")
    print(f"Resuming: {rows_done:,}/{N:,} rows already embedded ({rows_done / N:.1%})")
else:
    rows_done, emb = 0, None
    print(f"Starting fresh: {N:,} rows to embed")

session_start = rows_done
t0 = time.time()
for start in range(rows_done, N, BATCH_SIZE):
    batch = texts[start:start + BATCH_SIZE]
    vectors = np.asarray(embed_batch(batch), dtype=np.float32)

    if emb is None:   # dimension is only known after the first response
        emb = np.lib.format.open_memmap(
            OUT_PATH, mode="w+", dtype=np.float32, shape=(N, vectors.shape[1]))
        print(f"Created {OUT_PATH.name} with shape ({N:,}, {vectors.shape[1]})")

    emb[start:start + len(batch)] = vectors
    emb.flush()

    rows_done = start + len(batch)
    PROG_PATH.write_text(json.dumps({"rows_done": rows_done, "model": MODEL}))

    batch_no = start // BATCH_SIZE + 1
    if batch_no % 10 == 0 or rows_done == N:
        elapsed = time.time() - t0
        rate = (rows_done - session_start) / max(elapsed, 1e-9)
        eta = (N - rows_done) / max(rate, 1e-9)
        print(f"  {rows_done:,}/{N:,} ({rows_done / N:.1%})  "
              f"elapsed {elapsed / 60:.1f} min  ETA {eta / 60:.1f} min", flush=True)

print(f"\nDone — {rows_done:,} rows embedded in {(time.time() - t0) / 60:.1f} min")

Starting fresh: 384,756 rows to embed
Created node_embeddings.npy with shape (384,756, 1024)
  1,280/384,756 (0.3%)  elapsed 0.2 min  ETA 65.1 min
  2,560/384,756 (0.7%)  elapsed 0.5 min  ETA 74.2 min
  3,840/384,756 (1.0%)  elapsed 0.8 min  ETA 81.7 min
  5,120/384,756 (1.3%)  elapsed 1.0 min  ETA 75.9 min
  6,400/384,756 (1.7%)  elapsed 1.2 min  ETA 68.6 min
  7,680/384,756 (2.0%)  elapsed 1.3 min  ETA 66.1 min
  8,960/384,756 (2.3%)  elapsed 1.5 min  ETA 62.9 min
  10,240/384,756 (2.7%)  elapsed 1.7 min  ETA 61.2 min
  11,520/384,756 (3.0%)  elapsed 1.8 min  ETA 58.9 min
  12,800/384,756 (3.3%)  elapsed 2.0 min  ETA 57.1 min
  14,080/384,756 (3.7%)  elapsed 2.1 min  ETA 55.4 min
  15,360/384,756 (4.0%)  elapsed 2.2 min  ETA 54.0 min
  16,640/384,756 (4.3%)  elapsed 2.4 min  ETA 52.0 min
  17,920/384,756 (4.7%)  elapsed 2.5 min  ETA 51.9 min
  19,200/384,756 (5.0%)  elapsed 2.7 min  ETA 51.8 min
  20,480/384,756 (5.3%)  elapsed 2.9 min  ETA 51.1 min
  21,760/384,756 (5.7%)  elapsed 3

In [5]:
saved = np.load(OUT_PATH, mmap_mode="r")
print(f"{OUT_PATH}")
print(f"  shape : {saved.shape}")
print(f"  dtype : {saved.dtype}")
print(f"  size  : {OUT_PATH.stat().st_size / 1e9:.2f} GB")
print(f"  all-zero rows (never written): {int((np.abs(saved).sum(axis=1) == 0).sum()):,}")

/Users/gemacsanad/egyetem/DyTAG/baseline/joint_dataset/node_embeddings.npy
  shape : (384756, 1024)
  dtype : float32
  size  : 1.58 GB
  all-zero rows (never written): 0
